# Task B: private BraTS 2021 glioma segmentation research
Four modalities in order T1/T1ce/T2/FLAIR; release labels 0/1/2/4.
No verified real cohort is included. Masks, provenance and authorized access must be
independently reviewed. Empty masks never imply Task A normality. No Task C changes.
Keep the notebook private, disable saving outputs, and never share scans, manifests or weights.
Build scripts/build_task_a_bundle.py (now bundles both tasks), upload the code-only ZIP,
and retain the SHA-256 independently. Select Python 3.12 and GPU for synthetic_gpu or verified.
After Restart session, rerun from the first cell; no manual CODE_ROOT restoration is needed.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive').resolve()
BUNDLE_PATH = DRIVE_BASE / 'reviewed_code/task-ab-code.zip'  # Set your code-only ZIP.
BUNDLE_SHA256 = ''  # Required SHA-256 recorded independently when building the ZIP.
CONFIG_PATH = None  # Later: Path to your PRIVATE JSON config; never print its contents.


In [ ]:
import hashlib
import json
import stat
import zipfile

if (len(BUNDLE_SHA256) != 64 or any(c not in '0123456789abcdef' for c in BUNDLE_SHA256)
        or not BUNDLE_PATH.resolve().is_relative_to(DRIVE_BASE)
        or not BUNDLE_PATH.is_file() or BUNDLE_PATH.stat().st_size > 20 * 1024 * 1024):
    raise RuntimeError('Configure the private reviewed code bundle and independent SHA-256.')
blob = BUNDLE_PATH.read_bytes()
if hashlib.sha256(blob).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError('Code bundle checksum mismatch.')
CODE_ROOT = Path('/content') / ('task_ab_code_' + BUNDLE_SHA256[:16])
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    entries = archive.infolist()
    names = [entry.filename for entry in entries]
    if (len(entries) > 64 or len(set(names)) != len(names)
            or sum(entry.file_size for entry in entries) > 20 * 1024 * 1024):
        raise RuntimeError('Code bundle resource or duplicate-entry check failed.')
    for entry in entries:
        name = Path(entry.filename)
        if (name.is_absolute() or '..' in name.parts or '\\' in entry.filename
                or stat.S_ISLNK(entry.external_attr >> 16) or entry.file_size > 2 * 1024 * 1024):
            raise RuntimeError('Unsafe code bundle member.')
    manifest = json.loads(archive.read('code_manifest.json'))
    if (manifest.get('code_only') is not True or manifest.get('schema_version') != 1
            or set(names) != set(manifest['files']) | {'code_manifest.json'}):
        raise RuntimeError('Code-only allowlist check failed.')
    # Persist the verified manifest too; restart cells need it on disk.
    verified_files = {**manifest['files'], 'code_manifest.json':
                      hashlib.sha256(archive.read('code_manifest.json')).hexdigest()}
    for name, expected in verified_files.items():
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != expected:
            raise RuntimeError('Bundled source checksum mismatch.')
        target = CODE_ROOT / name
        if any(parent.is_symlink() for parent in (target, *target.parents)):
            raise RuntimeError('Code destination symlink rejected.')
        if target.exists():
            if target.read_bytes() != content:
                raise RuntimeError('Existing code snapshot differs; use a fresh runtime.')
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
print({'reviewed_code_verified': True})

STATE_PATH = Path('/content/brainagenticthon-code-state.json')
STATE_PATH.write_text(json.dumps({'code_root': str(CODE_ROOT), 'bundle_sha256': BUNDLE_SHA256,
    'manifest_sha256': hashlib.sha256(
        (CODE_ROOT / 'code_manifest.json').read_bytes()).hexdigest()}))


In [ ]:
# ruff: noqa: E402
import hashlib
import json
import sys
from pathlib import Path

# Survives Restart session: recover and revalidate the reviewed snapshot.
state = json.loads(Path('/content/brainagenticthon-code-state.json').read_text())
CODE_ROOT = Path(state['code_root'])
manifest_blob = (CODE_ROOT / 'code_manifest.json').read_bytes()
if hashlib.sha256(manifest_blob).hexdigest() != state['manifest_sha256']:
    raise RuntimeError('Reviewed manifest changed; rerun bundle verification.')
manifest = json.loads(manifest_blob)
for name, expected in manifest['files'].items():
    target = CODE_ROOT / name
    if (Path(name).is_absolute() or '..' in Path(name).parts
            or target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
            or hashlib.sha256(target.read_bytes()).hexdigest() != expected):
        raise RuntimeError('Reviewed snapshot changed; rerun bundle verification.')
import subprocess

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('This pinned research environment requires Python 3.12.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r',
                str(CODE_ROOT / 'requirements-task-a.txt')], check=True)
sys.path.insert(0, str(CODE_ROOT / 'src'))

# Check this research profile; Colab may contain unrelated incompatible packages.
from importlib.metadata import version

for requirement in (CODE_ROOT / 'requirements-task-a.txt').read_text().splitlines():
    if requirement.strip() and not requirement.startswith('#'):
        package, expected = requirement.strip().split('==')
        if version(package).split('+')[0] != expected:
            raise RuntimeError('Installed research dependency differs from reviewed pin.')


In [ ]:
# ruff: noqa: E402
import hashlib
import json
import sys
from pathlib import Path

# Survives Restart session: recover and revalidate the reviewed snapshot.
state = json.loads(Path('/content/brainagenticthon-code-state.json').read_text())
CODE_ROOT = Path(state['code_root'])
manifest_blob = (CODE_ROOT / 'code_manifest.json').read_bytes()
if hashlib.sha256(manifest_blob).hexdigest() != state['manifest_sha256']:
    raise RuntimeError('Reviewed manifest changed; rerun bundle verification.')
manifest = json.loads(manifest_blob)
for name, expected in manifest['files'].items():
    target = CODE_ROOT / name
    if (Path(name).is_absolute() or '..' in Path(name).parts
            or target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
            or hashlib.sha256(target.read_bytes()).hexdigest() != expected):
        raise RuntimeError('Reviewed snapshot changed; rerun bundle verification.')
sys.path.insert(0, str(CODE_ROOT / 'src'))
import nibabel as nib
import numpy as np
import torch

import brainagenticthon
from brainagenticthon.detection.training import private_output_directory
from brainagenticthon.preprocessing.oasis2_mri_quality import private_call, private_path

if not Path(brainagenticthon.__file__).resolve().is_relative_to(CODE_ROOT):
    raise RuntimeError('Previously imported project differs from reviewed bundle; restart runtime.')
if (torch.__version__.split('+')[0] != '2.6.0'
        or np.__version__ != '2.5.4' or nib.__version__ != '5.4.2'):
    raise RuntimeError('Runtime imports differ from pinned packages; restart the runtime.')
print({'dependencies_verified': True, 'cuda_available': torch.cuda.is_available()})

from brainagenticthon.segmentation.data import SyntheticCohort
from brainagenticthon.segmentation.training import (
    TrainConfig,
    evaluate_test,
    fit,
    load_private_cohort,
)


In [ ]:
if CONFIG_PATH is None:
    CONFIG = json.loads((CODE_ROOT / 'configs/task_b_research.example.json').read_text())
else:
    if not Path(CONFIG_PATH).resolve().is_relative_to(DRIVE_BASE):
        raise RuntimeError('Use authorized private Drive configuration.')
    CONFIG = private_call(lambda: json.loads(Path(CONFIG_PATH).read_text()))
MODE = CONFIG['mode']
if MODE not in {'synthetic', 'synthetic_gpu', 'verified'}:
    raise RuntimeError('Explicit mode required.')
PRIVATE_ROOT = Path(CONFIG['private_drive_root']).resolve()
if PRIVATE_ROOT == DRIVE_BASE or not PRIVATE_ROOT.is_relative_to(DRIVE_BASE):
    raise RuntimeError('Dedicated private Drive root required.')
OUTPUT = private_path(PRIVATE_ROOT, CONFIG['output_run'])
if 'REPLACE_WITH' in OUTPUT.name:
    raise RuntimeError('Set a fresh private output_run.')
settings = {**CONFIG['training'], 'patch': tuple(CONFIG['training']['patch'])}
if MODE == 'verified':
    if CONFIG.get('real_training_enabled') is not True or settings['device'] != 'cuda':
        raise RuntimeError('Real training requires explicit approval and CUDA.')
else:
    settings.update(device='cuda' if MODE == 'synthetic_gpu' else 'cpu', patch=(16, 16, 16))
    torch.set_num_threads(1)
TRAIN = TrainConfig(**settings)
if TRAIN.device == 'cuda':
    from brainagenticthon.detection.training import cuda_smoke_test
    print(cuda_smoke_test())


In [ ]:
if MODE == 'verified':
    COHORT = load_private_cohort(private_path(PRIVATE_ROOT, CONFIG['manifest']),
                                  private_path(PRIVATE_ROOT, CONFIG['image_root']),
                                  private_path(PRIVATE_ROOT, CONFIG['evidence']))
else:
    COHORT = SyntheticCohort()
private_output_directory(OUTPUT, COHORT.source_roots)
print({'eligibility_gate_passed': True, 'synthetic_only': MODE != 'verified'})


In [ ]:
if any(type(CONFIG.get(k)) is not bool for k in ('resume', 'trusted_resume')):
    raise RuntimeError('Explicit boolean resume settings required.')
MODEL, HISTORY, BOUND = private_call(
    fit, COHORT, TRAIN, OUTPUT, resume=CONFIG['resume'],
    trusted_resume=CONFIG['trusted_resume'], on_epoch=lambda row: print(row))
print({'completed_epochs': len(HISTORY), 'device': TRAIN.device, 'clinical_approval': False})


In [ ]:
if type(CONFIG.get('final_test_evaluation')) is not bool:
    raise RuntimeError('Explicit final evaluation boolean required.')
if CONFIG['final_test_evaluation']:
    print(private_call(evaluate_test, MODEL, COHORT, TRAIN, OUTPUT, BOUND, confirm=True))


For private inference use `load_case(root, row, require_mask=False)` with exactly four
reviewed modality paths and hashes, then `sliding_window(MODEL, image, patch=TRAIN.patch,
device=TRAIN.device).argmax(0)`. `release_labels` maps internal class 3 back to release
label 4. Use the returned affine to save a NIfTI only to a private output directory.
`lesion_boxes(mask, source="predicted_mask")` returns actual nonzero connected-component
bounds in the supplied image grid. These are model predictions, not expert annotations.
No registration/resampling occurs; the supplied grid and affine are preserved.
